# GNU Radio Conference 2026 CTF: Excavation Troubles

You are investigating an archeological dig site, when you stumble upon a dark room  with a single gravestone in the middle. As you turn around to leave, the door slides shut. Yikes!

Taking a closer look at the gravestone, it reads:
> *Here lies a faithful companion*
>
> *Who brought the world to our living room*
>
> *1896-1973*
>
> *Forever tuned to our hearts*
>
> 📻
>
> **What surfaces when precision fails?**

...There's a radio buried here? You quickly uncover the buried radio and a laptop! You gotta get out of here, and you've never been good at riddles.

The laptop contains five SigMF recordings of different signal classes. Use the released TorchSig Models XCiT classifier to identify each recording, then concatenate the first character of each predicted class in capture order.

**Run the cells below to load the classifier and begin the investigation.**

*Highly recommend changing runtime to T4 GPU.*

## 1. Imports and paths

The checkpoint is downloaded from the official TorchSig Models v1.0.0 release if it is not already present. Your file structure should look like:
```bash
.
├── captures
│   ├── capture_01.sigmf-data
│   ├── capture_01.sigmf-meta
│   ├── capture_02.sigmf-data
│   ├── capture_02.sigmf-meta
│   ├── capture_03.sigmf-data
│   ├── capture_03.sigmf-meta
│   ├── capture_04.sigmf-data
│   ├── capture_04.sigmf-meta
│   ├── capture_05.sigmf-data
│   ├── capture_05.sigmf-meta
│   └── metadata.csv
├── generate_captures.py
├── README.md
├── requirements.txt
└── xcit_narrowband_v1.0.0.ckpt
```

In [ ]:
!curl -sL https://github.com/TorchDSP/torchsig/releases/download/v2.2.0/grcon26-assets.zip -o assets.zip
!unzip assets.zip
!mv grcon26-assets/ctf/* .
!rm -rf ctf grcon26-assets
!rm assets.zip

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy import signal
from torchsig.datasets.datasets import StaticTorchSigDataset
from torchsig.utils.data_loading import WorkerSeedingDataLoader
from torchsig.utils.file_handlers import SigMFReader

from torchsig_models.models import XCiTClassifier

HERE = Path.cwd()
CAPTURE_DIR = HERE / 'captures'
CHECKPOINT = HERE / 'xcit_narrowband_v1.0.0.ckpt'
CHECKPOINT_URL = (
    'https://github.com/TorchDSP/torchsig-models/releases/download/'
    'v1.0.0/xcit_narrowband_v1.0.0.ckpt'
)
if not CHECKPOINT.exists():
    print('Downloading the official pretrained XCiT checkpoint...')
    urlretrieve(CHECKPOINT_URL, CHECKPOINT)
print(f'Checkpoint: {CHECKPOINT}')

## 2. Load and inspect the SigMF recordings

SigMF keeps the complex sample stream in `.sigmf-data` and recording metadata in `.sigmf-meta`. A TorchSig `StaticTorchSigDataset` uses `SigMFReader` as its file handler, and `WorkerSeedingDataLoader` batches the recordings. The class label is deliberately absent; `metadata.csv` contains only placeholder labels required by the dataset reader.

In [ ]:
meta_paths = sorted(CAPTURE_DIR.glob('capture_*.sigmf-meta'))
dataset = StaticTorchSigDataset(
    root=CAPTURE_DIR,
    file_handler_class=SigMFReader,
    target_labels=[],
)
dataloader = WorkerSeedingDataLoader(
    dataset, batch_size=len(dataset), shuffle=False, num_workers=0, seed=0
)
iq_batch = next(iter(dataloader))
iq_samples = iq_batch.numpy()

fig, axes = plt.subplots(len(iq_samples), 1, figsize=(10, 10), constrained_layout=True)
for path, metadata, iq, axis in zip(meta_paths, dataset.reader.sigmf_metadata, iq_samples, axes):
    sample_rate = metadata['global']['core:sample_rate']
    axis.specgram(iq, NFFT=256, Fs=sample_rate, noverlap=192)
    axis.set_title(path.stem)
    axis.set_ylabel('Hz')
axes[-1].set_xlabel('Time (s)')
plt.show()

## 3. Restore the pretrained model

The release checkpoint was trained with TorchSig 2.1.1. Its 57 outputs follow the signal-generator registry order below. This historical order must be retained even when the notebook runs with a newer TorchSig installation.

In [ ]:
CLASS_NAMES = [
    'tone', 'ofdm-64', 'ofdm-72', 'ofdm-128', 'ofdm-180', 'ofdm-256',
    'ofdm-300', 'ofdm-512', 'ofdm-600', 'ofdm-900', 'ofdm-1024',
    'ofdm-1200', 'ofdm-2048', 'lfm-data', 'lfm-radar', '2fsk', '4fsk',
    '8fsk', '16fsk', '2gfsk', '4gfsk', '8gfsk', '16gfsk', '2msk',
    '4msk', '8msk', '16msk', '2gmsk', '4gmsk', '8gmsk', '16gmsk',
    'fm', 'ook', 'bpsk', 'qpsk', '8psk', '16psk', '32psk', '64psk',
    '4ask', '8ask', '16ask', '32ask', '64ask', '16qam', '32qam',
    '64qam', '256qam', '1024qam', '32qam_cross', '128qam_cross',
    '512qam_cross', 'chirpss', 'am-dsb', 'am-dsb-sc', 'am-usb', 'am-lsb',
]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = XCiTClassifier.load_from_checkpoint(CHECKPOINT, map_location=device)
model.to(device).eval()
print(f'Loaded {len(CLASS_NAMES)}-class XCiT on {device}')

## 4. Classify the captures

`ComplexTo2D`, used during TorchSig training, represents complex IQ as two real channels. The equivalent conversion below produces a `[batch, 2, samples]` tensor.

In [ ]:
# Stack all IQ samples into a single batched tensor for the model.
# Each IQ sample contributes its real and imaginary parts (2 "channels"),
# then all samples are stacked along a new batch dimension → [batch, 2, length]
model_input = torch.from_numpy(
    np.stack([np.stack((iq.real, iq.imag)) for iq in iq_samples])
).to(device=device, dtype=torch.float32)

# Run inference: no gradient tracking (faster, less memory).
with torch.inference_mode():
    probabilities = model(model_input).softmax(dim=1).cpu()

In [ ]:
# ===== TorchSig CTF Challenge: Part 1 =====
# Your model outputs a tensor of probabilities (one probability per class for
# every sample). Your job is to turn those raw numbers back into human-readable
# labels so the results can actually be reported.
#
# Write a loop that iterates over the paired inputs and outputs using zip():
#   - `meta_paths`   : the list of original signal-file paths (one per sample)
#   - `probabilities`: the tensor from the previous cell (one row per sample)
#
# For each pair you must:
#   1. Find the predicted class by locating the index of the maximum probability
#      in that sample's score row (hint: .max(dim=0)).
#   2. Convert that numeric class index into a readable class name using the
#      CLASS_NAMES label table.
#   3. Append that class name to the predictions list.
#
# Expected result: predictions should be a list of strings of predicted signal
# class names, for example:
#   predictions = ['SSB', 'AM', 'FSK']
#
# Do NOT modify the model_input or probabilities cells above — only fill in the
# loop body below.
# ============================================================
predictions = []
for path, scores in zip(meta_paths, probabilities):
    # === ADD YOUR CODE BELOW ===

    pass

    # === ADD YOUR CODE ABOVE ===

In [ ]:
predictions

## 5. Recover the flag

Take the first character of every predicted class, preserving capture order.

In [ ]:
# ===== TorchSig CTF Challenge: Part 2 =====
# Now that you have a list of class names in `predictions`, you need to turn
# them into a single compact answer string.
#
# Build `answer` by taking the FIRST letter of every class name in order,
# joining them together, and converting the result to UPPERCASE.
#
#
# Example:
#   predictions = ['AM', 'FM', 'SSB', 'CW']
#   → answer = 'AFSC'
#
# Fill in the line below so that `answer` correctly assembles the flag from
# the predicted classes, then run the cell to print it.
# ============================================================

# === ADD YOUR CODE BELOW ===

answer = ""

# === ADD YOUR CODE ABOVE ===

print(f'CTF answer: {answer}')